In [ ]:
import tensorrt as trt
import onnxruntime as ort

print(f"TensorRT Version: {trt.__version__}")
print(f"Available Providers: {ort.get_available_providers()}")

# If 'TensorrtExecutionProvider' is in the list, you are golden!

TensorRT Version: 10.15.1.29
Available Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [ ]:
import cv2
import logging
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
# import subprocess
# import os
# import threading
# import queue
# import onnxruntime as ort

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

TensorFlow GPU Memory Growth Enabled


In [ ]:
def load_smart_session(model_path):
    # Order of priority for 2026 hardware
    # 1. TensorRT (NVIDIA)
    # 2. OpenVINO (Intel CPU/GPU/NPU)
    # 3. CPU (Generic Fallback)
    
    try:
        # TRIES NVIDIA FIRST
        print("Attempting to load TensorRT...")
        providers = [
            ('TensorrtExecutionProvider', {
                'device_id': 0,
                'trt_fp16_enable': True,       # Half precision for 2x speed
                'trt_engine_cache_enable': True, # Skips rebuild on next run
                'trt_engine_cache_path': './cache'
            }),
            'CUDAExecutionProvider'
        ]
        session = ort.InferenceSession(model_path, providers=providers)
        
    except Exception as e:
        print(f"TensorRT failed or not found. Switching to Intel/OpenVINO... Error: {e}")
        try:
            # TRIES INTEL NEXT
            providers = [
                ('OpenVINOExecutionProvider', {
                    'device_type': 'AUTO',      # Automatically picks GPU/NPU/CPU
                    'precision': 'FP16'
                })
            ]
            session = ort.InferenceSession(model_path, providers=providers)
            
        except Exception:
            # ULTIMATE FALLBACK
            print("No hardware acceleration found. Using standard CPU.")
            session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

    print(f"✅ Session loaded using: {session.get_providers()[0]}")
    return session

# Usage
session = load_smart_session("yolo26m_19.onnx")

# session = ort.InferenceSession("yolo26m.onnx", providers=['CPUExecutionProvider'])

Attempting to load TensorRT...
✅ Session loaded using: TensorrtExecutionProvider


In [ ]:
cap = cv2.VideoCapture('video/5.mp4')

In [ ]:
# 2. Define Color Ranges (Hue, Saturation, Value)
# Red has two ranges because it wraps around the 0/180 degree mark
lower_red1 = np.array([0, 100, 100])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([160, 100, 100])
upper_red2 = np.array([180, 255, 255])

lower_yellow = np.array([15, 100, 100])
upper_yellow = np.array([35, 255, 255])

lower_green = np.array([40, 100, 100])
upper_green = np.array([90, 255, 255])

# Configuration
CLASS_NAMES = ['Green', 'Red', 'Yellow'] # Ensure this matches your training order
COLOR_MAP = {'Green': (0, 255, 0), 'Red': (0, 0, 255), 'Yellow': (0, 255, 255)}

In [ ]:
def get_light_state(crop):
    # 1. Convert to HSV
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    
    

    # 3. Create Masks
    mask_red = cv2.addWeighted(cv2.inRange(hsv, lower_red1, upper_red1), 1.0, 
                               cv2.inRange(hsv, lower_red2, upper_red2), 1.0, 0)
 
    mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_green = cv2.inRange(hsv, lower_green, upper_green)

    # 4. Divide crop into 3 horizontal zones (Top, Middle, Bottom)
    height, _ = mask_red.shape
    top = mask_red[0:height//3, :]
    mid = mask_yellow[height//3:2*height//3, :]
    bot = mask_green[2*height//3:height, :]

    # 5. Determine state based on pixel density in specific zones
    if cv2.countNonZero(top) > 10: # Threshold of 10 pixels
        return "Red"
    elif cv2.countNonZero(mid) > 10:
        return "Yellow"
    elif cv2.countNonZero(bot) > 10:
        return "Green"
    
    return "UNKNOWN"

In [ ]:
# Get metadata once
input_name = session.get_inputs()[0].name
output_names = [output.name for output in session.get_outputs()]
input_shape = session.get_inputs()[0].shape # e.g., [1, 3, 640, 640]

print('input name '+ input_name)
print('output name ')
print(output_names)
print('input shape ')
print( input_shape)

input name images
output name 
['output0']
input shape 
['batch', 3, 'height', 'width']


In [ ]:
OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720

In [ ]:
while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    # --- 2. PREPROCESS ---
    h, w = frame.shape[:2]    #move it outside for efficeiny if camera height and width is constant
    # Resize to 640x640 (standard YOLO26m input)
    blob = cv2.resize(frame, (640, 640))
    blob = blob.astype(np.float32) / 255.0  # Normalize
    blob = blob.transpose(2, 0, 1)          # HWC to CHW
    blob = np.expand_dims(blob, axis=0)     # Add batch dim


    # --- 3. INFERENCE ---
    # YOLO26m is NMS-Free: the output is already the 'final' set of boxes
    outputs = session.run(output_names, {input_name: blob})


    
    # Output shape is typically [1, 300, 6] -> [batch, detections, x1,y1,x2,y2,score,class]
    predictions = outputs[0][0] 

    # --- SPEED HACK: Vectorized Filtering ---
    # Create a mask for rows where class is 9 and score is > 0.45
    mask = (predictions[:, 5] == 9) & (predictions[:, 4] > 0.30)
    traffic_lights = predictions[mask]
    # print(traffic_lights)
    

    # --- 4. DRAWING & FILTERING ---
    for light in traffic_lights:
        x1, y1, x2, y2,score,cls= light
        
        # if score > 0.30 and int(cls) ==9:  # Confidence threshold
            # Scaling back from 640x640 to original frame resolution
        ix1, iy1 = int(x1 * (w / 640)), int(y1 * (h / 640))
        ix2, iy2 = int(x2 * (w / 640)), int(y2 * (h / 640))
        
        # Label based on class index (Class 9 = Traffic Light)
        # label = f"Traffic Light: {score:.2f}" if int(cls) == 9 else f"Obj {int(cls)}"
        color = (0, 255, 0)
        
        # Draw Box
        cv2.rectangle(frame, (ix1, iy1), (ix2, iy2), color, 2)
        # cv2.putText(frame, label, (ix1, iy1 - 10), 
        #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # --- 5. DISPLAY ---
    display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imshow("YOLO26m NMS-Free Production", display_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# frame_count = 0

# OUTPUT_WIDTH = 1280
# OUTPUT_HEIGHT = 720

# print('before')

# input_name = session.get_inputs()[0].name
# output_names = [output.name for output in session.get_outputs()]

# print('here')

# while cap.isOpened() and frame_count ==0:
#     success, frame = cap.read()
#     if not success:
#         break
#     print('1')
#     frame_count += 1

#     if frame_count % 2 != 0:
#         # display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
#         # cv2.imshow('Traffic Light Monitor', display_frame) # Keep the window fluid
#         # if cv2.waitKey(1) & 0xFF == ord('q'): break
#         continue

#     # --- PREPROCESS (Fastest way) ---
#     # Resize to 640x640, convert BGR to RGB, normalize to 0.0-1.0
#     img = cv2.resize(frame, (640, 640))
#     img = img.astype(np.float32) / 255.0
#     img = img.transpose(2, 0, 1)  # HWC to CHW
#     img = np.expand_dims(img, axis=0) # Add batch dimension
    

#     # --- INFERENCE ---
#     outputs = session.run(output_names, {input_name: img})

#     # --- POST-PROCESS (Class 9 = Traffic Light) ---
#     # In 2026, many models exported with end-to-end NMS 
#     # return outputs in [num_dets, 6] format: [x1, y1, x2, y2, score, class]
#     detections = outputs[0]

#     print(detections)
    
    
#     # if len(detections) > 0:
#     #     valid_crops = []
#     #     valid_boxes = []
       

#     #     # best_box = max(lights, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
#     #     for box in detections:
#     #         x1, y1, x2, y2,score,cls = det
    
#     #         # 5. Crop and Classify with your Vertical CNN
#     #         # Note: Ensure crop is not empty
#     #         crop = frame[max(0, y1):y2, max(0, x1):x2]
#     #         if crop.size > 0:
#     #             # Resize to your new vertical shape (64 width, 64 height)
#     #             # crop_resized = cv2.resize(crop, (64, 64)) / 255.0

#     #             crop_resized = cv2.resize(crop, (64, 64))  #removed for hsv

#     #             valid_crops.append(crop_resized)
#     #             valid_boxes.append((x1, y1, x2, y2))

#     #     # 2. Perform Batch Inference (The Speed Hack)
#     #     if valid_crops:

#     #         # # Convert list to a Tensor (this is faster than np.array for the function call)
#     #         # batch_tensor = tf.convert_to_tensor(valid_crops, dtype=tf.float32)
            
    
#     #         # # Calling the model directly skips the heavy .predict() wrapper
#     #         # predictions = classifier(batch_tensor, training=False).numpy()

#     #         # 3. Draw UI
#     #         for i, (x1, y1, x2, y2) in enumerate(valid_boxes):
#     #             # pred_scores = predictions[i]
#     #             # color_idx = np.argmax(pred_scores)
#     #             # conf = pred_scores[color_idx]
#     #             # color_label = CLASS_NAMES[color_idx]

#     #             color_label = get_light_state(valid_crops[i])
                
#     #             bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
#     #             cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
#     #             cv2.putText(frame, f"{color_label}", (x1, y1-10), 
#     #                     cv2.FONT_HERSHEY_SIMPLEX, 0.9, bgr_color, 2)
                
#     # display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
#     # cv2.imshow('Traffic Light Monitor', display_frame)
#     # # Exit on 'q' key
#     # if cv2.waitKey(1) & 0xFF == ord('q'):
#     #     break
    
    

# cap.release()
# cv2.destroyAllWindows()